In [3]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg    import lstsq, solve, qr, svd
from scipy.optimize  import fsolve
from scipy.integrate import quad, solve_ivp
from scipy.special   import roots_legendre
from sympy import symbols, Matrix, eye, lambdify, nsimplify

In [9]:
def stability_function():
    z = symbols("z")
    U = Matrix([[0,0,0], [0,0,0], [0,0,0]])
    b = Matrix([1/3, 1/3, 1/3]).T 
    S_ = sum(b*(eye(3) - z*U).inv())
    S = 1 + z * S_
    print(nsimplify(S))
    return lambdify(z,S)
stability_function()

z + 1


<function _lambdifygenerated(z)>

In [ ]:
# war das hier nur zum üben

def compute_weights(a,b,s,c):
    V = np.vander(c, N=s, increasing=True).T
    rhs = np.array([(b**(k+1) - a**(k+1)) / (k+1) for k in range(s)])
    w = np.linalg.solve(V,rhs)
    return c, w 

def compute_weights_and_nodes(a,b,s):
    def eqs(vars):
        x1, x2, w1, w2 = vars
        rhs = [1, 1/2, 1/3, 1/4]
        return [w1 * x1**i + w2*x2**i - rhs[i] for i in range(len(rhs))]

    return fsolve(eqs, [0.5, 1.5, 1, 1 ])

def compute_order_and_genauigkeitsgrad(method, n, tol=1e-10):
    a, b = -1, 1
    c, w = method(n)

    for k in range(2*n +1):
        f = lambda x : x**k
        f_exact = (b**(k+1) - a**(k+1)) / (k+1)
        f_approx = np.sum(w*f(c))
        if abs(f_exact - f_approx) > tol:
            return k, k -1 

def stability_function():
    z = symbols("z")
    U = Matrix([[0,0,0,0], [1/2,0,0,0], [0, 1/4,1/4,0], [1/6, 2/6, 1/6,0]])
    b = Matrix([1/4, 1/4, 1/4, 1/4]).T 
    S_ = sum(b*(eye(4) - z*U).inv())
    S = 1 + z * S_
    print(nsimplify(S))
    return lambdify(z,S)

def h(S, lam = -3-3j):
    for h in np.linspace(0.001,1,1000):  
        if abs(S(lam * h))>1:
            return h 
        
def lstsqlincon(A, b, C, d):
    zero = np.zeros([C.shape[0], C.shape[0]])
    K = np.array([A.T@A, C.T],
                 [C, zero])
    rhs = np.concatenate([A.T@b, d])
    sol = lstsq(K, rhs)[0]
    m, n = A.shape
    return sol[:n]


def lstsqlincon(A, b, C, d):    
    zero = np.zeros((C.shape[0], C.shape[0]))
    K    = np.block([[A.T@A, C.T],
                     [C,   zero]])
    rhs  = np.hstack([A.T@b, d])
    sol  = lstsq(K, rhs)[0]
    m, n = A.shape
    return sol[:n]





## Quadrature

In [ ]:
# Pages: 14, 16, 17, 23

def compute_weights(a, b, s, c=np.linspace(a, b, s) else x):           
    V   = np.vander(c, N=s, increasing=True).T            # Vandermonde matrix of the nodes c
    rhs = np.array([(b**(k + 1) - a**(k + 1)) / (k + 1)   # exact integral of monomials x^p from a to b
                    for k in range(s) ])
    w   = np.linalg.solve(V, rhs)                         # w = M^{-1} * rhs   
    return c, w

def compute_weights_and_nodes(): # if both nodes and weights are not given
    def equations(vars):
        x1, x2, w1, w2 = vars
        rhs = [1, 0, 1/3, 0] # [-1,1]
        # rhs = [1, 1/2, 1/3, 1/4] # if [a,b] = [0,1]
        return [w1 * x1**i + w2 * x2**i - rhs[i] for i in range(len(rhs))]
    
    return fsolve(equations, [1, 1, 1, 1])

def compute_order_and_genauigkeitsgrad(method, n, tol = 1e-11): 
    a, b = -1, 1                                        # Interval for the quadrature rule
    c, w = method(n)

    for k in range(2*n + 1):
        f_ref    = lambda x: x ** k  
        I_exact  = (b**(k + 1) - a**(k + 1)) / (k + 1)  # = quad(f, a, b)[0] if [a,b] =! [-1,1]
        I_approx = np.sum(w * f_ref(c))
        if abs(I_approx - I_exact) > tol:
            return k, k-1 

def gauss(f, a, b, N):
    x, h = np.linspace(a, b, N + 1, retstep=True)
    [nodes, weights] = roots_legendre(5)

    I_gaus =    0.5 * (b - a) * weights @ f(0.5  * (b - a) * nodes + 0.5  * (a + b)) # Page: 28
    I_comp = np.sum([ 0.5 * h * weights @ f(xi + 0.5 * h * (nodes + 1))  for xi in x[:-1] ]) 
    return [I_gaus, I_comp]

In [ ]:
 # Example usage of gauss function
def gauss(f, a, b, N):
    x, h = np.linspace(a, b, N + 1, retstep=True)
    [nodes, weights] = roots_legendre(5)

    I_gaus =    0.5 * (b - a) * weights @ f(0.5  * (b - a) * nodes + 0.5  * (a + b)) # Page: 28
    I_comp = np.sum([ 0.5 * h * weights @ f(xi + 0.5 * h * (nodes + 1))  for xi in x[:-1] ]) 
    return [I_gaus, I_comp]

gauss(f= lambda x: x**2, a=0, b=1, N=10) 

[0.33333333333333326, 0.33333333333333337]

## Fourier

In [3]:
# Pages: 84

## ODE

In [ ]:
# Pages: 191, 206

# CONVERGENVE OF ODE METHODS
def compute_and_plot_exact_errors_rate(method_step):
    eval = 2 ** np.arange(4,15); T  = 10; N = 420; y0 = np.array([1.0, 0.0]) 
    rhs = lambda t, y: np.array([y[1], -9.81 / 0.6 * np.sin(y[0])]) # Example: Harmonic Oscillator

    # Compute: reference exact solution # Page 206
    ref_sol = solve_ivp(rhs, t_span=(0.0, T), y0=y0, method="RK45", atol=1e-16, rtol=1e-10) 
    t_exact = ref_sol.t; y_exact = ref_sol.y.T
    # Optional: t_eval = [10] get sol for t = 10 only

    # Plot: Method and Exact 
    t_approx, y_approx = integrate(method_step, rhs, y0, T, N)
    plt.plot(t_approx, y_approx[:, 0]) 
    plt.plot(t_exact , y_exact [:, 0]) 
    plt.show()

    # Compute and Plot: Errors and Rate 
    errors = [abs(y_exact[-1, 0] - integrate(method_step, rhs, y0, T, n)[1][-1, 0]) for n in eval]
    order   = - np.polyfit(np.log(eval), np.log(errors), deg=1) [0]
    plt.loglog(eval, errors)
    plt.loglog(eval, eval**order, label=f"conv order = {order}")

# SPLITTING METHODS
def Phi_A(y, dt):
    return [y[0] + dt * y[1], y[1]]
def Phi_B(y, dt):
    return [y[0]            , y[1] + dt * rhs(y[0], y[1])]

def splitting_method_step(y, dt, method = "SS"):
    a, b = splitting_parameters(method)   
    for ai, bi in zip(a, b):
        y = Phi_B(Phi_A(y, ai * dt), bi * dt)
    # y = Phi_A(y, dt)
    # y = Phi_B(y, dt)
    return y

def splitting_method(y0, t_end, N):
    return integrate(splitting_method_step, y0, t_end, N) 

# STABILITY FUNCTION
def stability_function(): # Page 272 theory only
    z = symbols("z")
    U = Matrix([[0,0,0,0],[1/2,0,0,0],[1/4,1/4,0,0],[0,-1,2,0]])
    b = Matrix([1/6, 0, 4/6, 1/6]).T
    S_ = sum(b * (eye(4) - z*U).inv())
    S = 1 + z * S_

    print(nsimplify(S))
    return lambdify(z, S) 

def basic_plot(S):
    x = np.linspace(-5, 5, 1000)
    X, Y = np.meshgrid(x, x)
    z = X + 1j*Y
    Z = np.abs(S(z)) < 1

    plt.contourf(X, Y, Z)
    plt.grid()

def compute_h(S, lam = -3 - 3j):
    for h in np.linspace(0.001, 1, 1000):
        z = h * lam
        if abs(S(z)) > 1:
            return h

## Nullstellensuche

In [ ]:
# Pages: 238, 247, 251, 253, 339

## Ausgleichgsrechnung

In [ ]:
# Pages: 299, 305, 323, 324

# Mit Nebenbedingungen
def lstsqlincon(A, b, C, d):    
    zero = np.zeros((C.shape[0], C.shape[0]))
    K    = np.block([[A.T@A, C.T],
                     [C,   zero]])
    rhs  = np.concatenate([A.T@b, d])
    sol  = lstsq(K, rhs)[0]
    m, n = A.shape
    return sol[:n]
    
    A = np.vander(x_main, N=degree+1, increasing=False)
    b = y_main
    C = np.vander(x_side, N=degree+1, increasing=False)
    d = y_side

    coeffs = lstsqlincon(A, b, C, d)

def Ausgleichsrechnung(x_gemessen, y_gemessen, degree): 
    A = np.vander(x_gemessen, N=degree+1, increasing=False)
    b = y_gemessen
    
    # Least Squares
    coeffs = lstsq(A, b)[0] 

    # QR:  Page: 299 and 324, c  =  R^-1 * Q.T * b
    Q, R   = qr(A) 
    coeffs = solve(R, Q.T @ b)

    # SVD:  Page: 305 and 323, c =  Vh.T * S * U.T * b
    U, S, Vh = svd(A)
    rA       = matrix_rank(A)
    coeffs   = Vh[:rA, :].T @ np.diag(1/S[:rA]) @ U[:, :rA].T @ b
    
    x_plot = np.linspace(min(x_gemessen), max(x_gemessen), 500)
    y_plot = np.polyval(coeffs, x_plot)
    
    plt.plot(x_plot, y_plot)
    plt.scatter(x_gemessen, y_gemessen)
    plt.show()
